In [ ]:
# load in from saved h5ad
import scanpy as sc
import pandas as pd

adata = sc.read_h5ad("../data/interim/01_adata_qc.h5ad")
print(adata)

In [ ]:
# parse the guide codes

## note, there are some guides i.e. ZBTB10_NegCtrl0__ZBTB10_NegCtrl0_2 where the "_2" is like a subset of guide0
## ignored this for now, may have been follow-up cells in the paper - check details

import re

# Use first half of guide_identity (verified redundant with second half,
# aside from an occasional trailing guide-variant suffix like "_1"/"_2"
# on the second copy -- confirmed present in the raw GEO file itself).
first_half = adata.obs["guide_identity"].astype(str).str.split("__").str[0]

# Split into the two guide "slots". Dual-guide vector: singles pair a real
# gene with a NegCtrl guide in the second slot; doubles have two real genes.
genes_split = first_half.str.split("_", n=1, expand=True)
genes_split.columns = ["gene_1", "gene_2"]

adata.obs["gene_1"] = genes_split["gene_1"]
adata.obs["gene_2"] = genes_split["gene_2"]

adata.obs[["guide_identity", "gene_1", "gene_2"]].head(10)

In [ ]:
# classify each cell (barcode) as ctrl, single, or double perturbation

def classify(row):
    g1_ctrl = row["gene_1"].startswith("NegCtrl")
    g2_ctrl = row["gene_2"].startswith("NegCtrl")
    if g1_ctrl and g2_ctrl:
        return "control"
    elif g1_ctrl or g2_ctrl:
        return "single"
    else:
        return "double"

adata.obs["perturbation_type"] = adata.obs.apply(classify, axis=1)
adata.obs["perturbation_type"].value_counts()

In [ ]:
# looks like 10k ctrl cells, you can average signal there and get a positive "washout" effect
# single gene perturbation: 53k divided by ~130 targets in this study = appx number cells/replicates per gene target

# code below to compute the estimate above

# For singles: which slot has the real gene varies per row (could be gene_1 or gene_2)
singles = adata.obs[adata.obs["perturbation_type"] == "single"].copy()
singles["target"] = singles.apply(
    lambda r: r["gene_1"] if not r["gene_1"].startswith("NegCtrl") else r["gene_2"],
    axis=1
)
print(singles["target"].nunique(), "distinct single-gene targets")
print(singles["target"].value_counts().describe()) # mean is number of cells/replicates per gene target

# note below that min replicates is 104 and max is 1.7k so filtering by robustness of "cluster" may be important before downstream analysis

In [ ]:
# build the target column, which is "control", "gene1", or "gene1,gene2"
# nonordinal (order does not have meaning) categorical variable (non-numeric)

def get_target(row):
    if row["perturbation_type"] == "control":
        return "control"
    elif row["perturbation_type"] == "single":
        return row["gene_1"] if not row["gene_1"].startswith("NegCtrl") else row["gene_2"]
    else:  # double
        return f"{row['gene_1']}+{row['gene_2']}"

adata.obs["target"] = adata.obs.apply(get_target, axis=1)
adata.obs["target"].value_counts().head(10)

In [ ]:
# save

adata.write("../data/interim/02_adata_parsedPerturb.h5ad", compression="gzip")